# ODS v2: Quote header

Builds 2 ods tables in `ods_v2`: `Quote`, `PolicyQuoteLink`.

Needs these notebooks to have succeeded first: 10_Policy.

Generated by `ods_build/v2/build_notebook.py` from `entities/*.yaml`: one SQL statement per cell.

**If a cell fails**

1. Read the failing cell's header: what it builds, what it reads, its grain, key and dedupe rule.
2. Fix the cause. Most failures are a source change (a renamed or missing column) or a broken key (duplicates).
   The checks cell at the end names the table and the likely cause.
3. Re-run from the failed cell: every cell rebuilds its table completely (`create or replace`), so re-running is
   always safe. After a session restart, run the first three cells again first (parameters, setup, start log).
4. **Copy the fix back into the config** (`ods_build/v2/entities/*.yaml`) and regenerate. This notebook is generated:
   a fix made only here is lost at the next generation (the drift check lists such cells).


In [ ]:
%%pyspark
# Parameters (set by ODS_v2_Run; leave blank when running by hand)
batch_id   = ""
source_set = ""


In [ ]:
%%pyspark
%run ODS_v2_00_Setup

In [ ]:
%%pyspark
ods_run_log("20_Quote", "started")


## Quote


In [ ]:
/******************************************************************************************************/
/*  Name        : ${ods.stg}.odsv2_mfq_query_rows                                                       */
/*  Description : MFQ quote query rows, contact details already hashed, times in UTC, only the columns  */
/*              : ods uses.                                                                             */
/*              :                                                                                       */
/*              : Reads: dlk.mfq_quotequery                                                             */
/******************************************************************************************************/
create or replace table ${ods.stg}.odsv2_mfq_query_rows as
select  q.QuoteQueryId,
        nullif(lower(trim(q.QuoteQueryGuid)), '')                                      as GuidKey,
        q.QuoteQueryGuid,
        q.CreatedDateTime,
        case when q.CreatedDateTime < timestamp '2026-06-12 00:00:00'
             then to_utc_timestamp(q.CreatedDateTime, 'Europe/Dublin')
             else q.CreatedDateTime end                             as CreatedDateTimeUtc,
        nullif(upper(trim(q.ReferenceCode)), '')                                       as ReferenceCode,
        nullif(lower(trim(q.RetrieveSessionToken)), '')                                as RetrieveSessionToken,
        q.RetrievalCount,
        q.StepId,
        case when upper(trim(q.ContactEmail)) = 'GDPR' or upper(trim(q.ContactMobile)) = 'GDPR' then 1 else 0 end
                                                                    as IsGdprWiped,
        ods_hash_email(q.ContactEmail)                                  as EmailKey,
        ods_hash_phone(q.ContactMobile)                                 as MobileKey,
        ods_hash_id(q.VehicleRegistrationNo, '')                            as VehicleRegKey,
        ods_eircode_area(q.AddressPostCode)                             as EircodeArea,
        ods_hash_id(concat_ws(' ', q.AddressLine1, q.AddressLine2), '')     as AddressKey,
        cast(q.CoverStartDate as date)                              as CoverStartDate,
        cast(q.PolicyStartDate as date)                             as PolicyStartDate,
        q.PolicyInceptionDate,
        q.BrokerID,
        q.Gclid,
        q.CoverCode,
        q.CoverVoluntaryExcess,
        q.NCDClaimedYearsEarned,
        q.NCDProtectionRequiredInd,
        q.NCDClaimedBonusProtectionType,
        q.PolicyPreviouslyInsuredInd,
        q.PolicyPreviousInsurer,
        q.ProposerSex,
        q.ProposerMaritalStatus,
        q.HomeownerInd,
        q.VehicleModelCode,
        q.VehicleModelName,
        q.VehicleManufacturedYear,
        q.VehicleFirstRegdYear,
        q.VehicleDateFirstRegistered,
        q.VehicleDatePurchased,
        q.VehicleCubicCapacity,
        q.VehicleBodyType,
        q.VehicleImported,
        q.VehicleSecurityDeviceInd,
        q.VehicleAnnualMilage,
        q.VehiclePleasureMileage,
        q.VehicleBusinessMileage,
        q.VehicleValue,
        q.VehicleAreaRating,
        q.VehicleAreaKeptAt,
        q.VehicleKeptAt,
        q.ServiceCallId,
        q.TestGroupFlag,
        q.VariantFlag,
        q.PricePointType
from    ${ods.dlk}.mfq_quotequery q
where   nullif(trim(q.QuoteQueryGuid), '') is not null

In [ ]:
/******************************************************************************************************/
/*  Name        : ${ods.stg}.odsv2_mfq_query_latest                                                     */
/*  Description : One row per MFQ guid, its latest query row, with first / last activity and row counts */
/*              : .                                                                                     */
/*              :                                                                                       */
/*              : Reads: stg.odsv2_mfq_query_rows                                                       */
/******************************************************************************************************/
create or replace table ${ods.stg}.odsv2_mfq_query_latest as
select  *
from    (
        select  r.*,
                row_number() over (partition by GuidKey order by CreatedDateTime desc, QuoteQueryId desc) as rn,
                max(QuoteQueryId)       over (partition by GuidKey)     as MaxQuoteQueryId,
                min(CreatedDateTimeUtc) over (partition by GuidKey)     as FirstCreatedDateTimeUtc,
                max(CreatedDateTimeUtc) over (partition by GuidKey)     as LastCreatedDateTimeUtc,
                count(*)                over (partition by GuidKey)     as QueryRowCount,
                max(ReferenceCode)      over (partition by GuidKey)     as AnyReferenceCode
        from    ${ods.stg}.odsv2_mfq_query_rows r
        ) x
where   rn = 1

In [ ]:
/******************************************************************************************************/
/*  Name        : ${ods.stg}.odsv2_mfq_fact_sale                                                        */
/*  Description : Sale and quoted price per MFQ guid from the warehouse MFQ fact, as notebook 02.       */
/*              :                                                                                       */
/*              : Reads: edw.tbl_fact_mfq                                                               */
/******************************************************************************************************/
create or replace table ${ods.stg}.odsv2_mfq_fact_sale as
select  nullif(lower(trim(QuoteQueryGuid)), '')                                        as GuidKey,
        max(SaleMFQ_GWP)                                            as SalePremium,
        max(nullif(upper(trim(PolicyCode)), ''))                                       as PolicyCode,
        max(Rank1CustPriceBOTH)                                     as QuotedPremium
from    ${ods.edw}.tbl_fact_mfq
where   nullif(trim(QuoteQueryGuid), '') is not null
group by nullif(lower(trim(QuoteQueryGuid)), '')

In [ ]:
/******************************************************************************************************/
/*  Name        : ${ods.stg}.odsv2_quote_ccm                                                            */
/*  Description : QuotationStorage (contact-centre motor) quotes, best returned premium, latest calcula */
/*              : tion.                                                                                 */
/*              :                                                                                       */
/*              : Reads: dlk.ext_quotationstorage_policydetails, dlk.ext_quotationstorage_riskdetails,  */
/*              : dlk.ext_quotationstorage_schemeresult                                                 */
/******************************************************************************************************/
create or replace table ${ods.stg}.odsv2_quote_ccm as
select  *
from    (
        select  cast(a.Quotation_ID as string)                      as Ref,
                a.Date_Calculated                                   as QuoteDateTime,
                cast(c.Policy_Start_Date as date)                   as CoverStartDate,
                s.BestPremium                                       as QuotedPremium,
                a.Accepted_Premium                                  as AcceptedPremium,
                nullif(upper(trim(a.Policy_Code)), '')                                 as SourcePolicyCode,
                row_number() over (partition by a.Quotation_ID order by a.Date_Calculated desc, a.`$ExtractDateTime` desc) as rn
        from    ${ods.dlk}.ext_quotationstorage_policydetails   a
        left join ${ods.dlk}.ext_quotationstorage_riskdetails   c   on c.Quotation_ID = a.Quotation_ID
        left join (
                select  Quotation_ID, min(Quote) as BestPremium
                from    ${ods.dlk}.ext_quotationstorage_schemeresult
                where   Decision = 'Quote'
                group by Quotation_ID
                ) s     on s.Quotation_ID = a.Quotation_ID
        ) x
where   rn = 1

In [ ]:
/******************************************************************************************************/
/*  Name        : ${ods.stg}.odsv2_quote_home_web                                                       */
/*  Description : HFQ (home web) quotes, latest version per quote reference, with the sold policy and p */
/*              : rice range.                                                                           */
/*              :                                                                                       */
/*              : Reads: dlk.hfq_quotedetails, dlk.hfq_sales, dlk.hfq_response_quotes                   */
/******************************************************************************************************/
create or replace table ${ods.stg}.odsv2_quote_home_web as
select  *
from    (
        select  nullif(upper(trim(a.QuoteCodeReference)), '')                          as Ref,
                a.CreatedOn                                         as QuoteDateTime,
                cast(a.policyDetails_startDate as date)             as CoverStartDate,
                c.MinPremium                                        as QuotedPremium,
                cast(null as decimal(18,2))                         as AcceptedPremium,
                b.PolicyCode                                        as SourcePolicyCode,
                row_number() over (partition by upper(trim(a.QuoteCodeReference))
                                   order by case when b.PolicyCode is not null then 0 else 1 end,
                                            a.UpdatedOn desc, a.InsertDateTime desc) as rn
        from    ${ods.dlk}.hfq_quotedetails a
        left join (
                select  upper(trim(QuoteCodeReference)) as Ref, max(nullif(upper(trim(PolicyCode)), '')) as PolicyCode
                from    ${ods.dlk}.hfq_sales
                where   Status = 1
                group by upper(trim(QuoteCodeReference))
                ) b     on b.Ref = upper(trim(a.QuoteCodeReference))
        left join (
                select  upper(trim(QuoteCodeReference)) as Ref, min(Quotes_TotalPayable) as MinPremium
                from    ${ods.dlk}.hfq_response_quotes
                where   Quotes_Outcome = 'PremiumReturned'
                group by upper(trim(QuoteCodeReference))
                ) c     on c.Ref = upper(trim(a.QuoteCodeReference))
        where   nullif(trim(a.QuoteCodeReference), '') is not null
        ) x
where   rn = 1

In [ ]:
/******************************************************************************************************/
/*  Name        : ${ods.stg}.odsv2_quote_home_cc                                                        */
/*  Description : Home QS (home, contact centre) quotes, latest transaction per quote (policy) id, chea */
/*              : pest premium.                                                                         */
/*              :                                                                                       */
/*              : Reads: dlk.ext_home_qs_policydetails, dlk.ext_home_qs_quotedetails                    */
/******************************************************************************************************/
create or replace table ${ods.stg}.odsv2_quote_home_cc as
select  *
from    (
        select  upper(a.PolicyId)                                   as Ref,
                a.TransactionDate                                   as QuoteDateTime,
                cast(a.EffectiveStartDate as date)                  as CoverStartDate,
                d.MinPremium                                        as QuotedPremium,
                cast(null as decimal(18,2))                         as AcceptedPremium,
                nullif(upper(trim(a.PolicyCode)), '')                                  as SourcePolicyCode,
                row_number() over (partition by upper(a.PolicyId) order by a.TransactionDate desc, a.`$ExtractDateTime` desc) as rn
        from    ${ods.dlk}.ext_home_qs_policydetails a
        left join (
                select  upper(p.PolicyId) as Ref, min(q.TotalPayable) as MinPremium
                from    ${ods.dlk}.ext_home_qs_policydetails p
                join    ${ods.dlk}.ext_home_qs_quotedetails  q   on q.HomeRiskId = upper(p.RiskId)
                where   q.Outcome = 'Premium Returned' and q.TotalPayable > 0
                group by upper(p.PolicyId)
                ) d     on d.Ref = upper(a.PolicyId)
        ) x
where   rn = 1

In [ ]:
/******************************************************************************************************/
/*  Name        : ${ods.stg}.odsv2_quote_travel                                                         */
/*  Description : Travel quotes, latest update per quote. quote_number 0 is an incomplete quote: its re */
/*              : ference is the hashed email
plus the travel start date.
                              */
/*              :                                                                                       */
/*              : Reads: dlk.ext_travel_quotes                                                          */
/******************************************************************************************************/
create or replace table ${ods.stg}.odsv2_quote_travel as
select  *
from    (
        select  case when q.quote_number <> 0 then cast(q.quote_number as string)
                     else concat('E:', ods_hash_email(q.email), ':', date_format(q.travel_startdate, 'yyyyMMdd'))
                end                                                 as Ref,
                q.travel_quoteDate                                  as QuoteDateTime,
                cast(q.travel_startdate as date)                    as CoverStartDate,
                q.travel_premium                                    as QuotedPremium,
                nullif(upper(trim(q.certificate_reference)), '')                       as CertificateReference,
                row_number() over (partition by case when q.quote_number <> 0 then cast(q.quote_number as string)
                                                     else concat(lower(trim(q.email)), date_format(q.travel_startdate, 'yyyyMMdd')) end
                                   order by q.LastUpdated desc, q.EXT_Travel_QuotesKey desc) as rn
        from    ${ods.dlk}.ext_travel_quotes q
        ) x
where   rn = 1
and     Ref is not null

In [ ]:
/******************************************************************************************************/
/*  Name        : ${ods.stg}.odsv2_quote_van                                                            */
/*  Description : Van quotes, latest version per quote reference.                                       */
/*              :                                                                                       */
/*              : Reads: dlk.vanquotedetails                                                            */
/******************************************************************************************************/
create or replace table ${ods.stg}.odsv2_quote_van as
select  *
from    (
        select  nullif(upper(trim(v.QuoteReference)), '')                              as Ref,
                to_timestamp(v.QuoteStartDateTime)                  as QuoteDateTime,
                to_date(v.CoverStartDate)                           as CoverStartDate,
                row_number() over (partition by upper(trim(v.QuoteReference)) order by v.CreateDateTime desc, v.PartitionKey desc) as rn
        from    ${ods.dlk}.vanquotedetails v
        where   nullif(trim(v.QuoteReference), '') is not null
        ) x
where   rn = 1

In [ ]:
/******************************************************************************************************/
/*  Name        : ${ods.stg}.odsv2_quote_all                                                            */
/*  Description : All quote systems in one shape, from the cut-off date.                                */
/*              :                                                                                       */
/*              : Reads: stg.odsv2_mfq_query_latest, stg.odsv2_mfq_fact_sale, stg.odsv2_quote_ccm, stg. */
/*              : odsv2_quote_home_web, stg.odsv2_quote_home_cc, stg.odsv2_quote_travel, stg.odsv2_quot */
/*              : e_van                                                                                 */
/******************************************************************************************************/
create or replace table ${ods.stg}.odsv2_quote_all as
select  2 as SourceSystemId, 'MFQ' as SourceSystem, 'Motor' as ProductType, m.GuidKey as Ref,
        m.FirstCreatedDateTimeUtc as QuoteDateTime, m.CoverStartDate, f.QuotedPremium, f.SalePremium as AcceptedPremium,
        f.PolicyCode as SourcePolicyCode, cast(null as string) as CertificateReference
from    ${ods.stg}.odsv2_mfq_query_latest m
left join ${ods.stg}.odsv2_mfq_fact_sale f on f.GuidKey = m.GuidKey
where   m.FirstCreatedDateTimeUtc >= date '${ods.cutoff_date}'
union all
select  9, 'QuotationStorage', 'Motor', Ref, QuoteDateTime, CoverStartDate, QuotedPremium, AcceptedPremium, SourcePolicyCode, null
from    ${ods.stg}.odsv2_quote_ccm where QuoteDateTime >= date '${ods.cutoff_date}'
union all
select  3, 'HFQ', 'Home', Ref, QuoteDateTime, CoverStartDate, QuotedPremium, AcceptedPremium, SourcePolicyCode, null
from    ${ods.stg}.odsv2_quote_home_web where QuoteDateTime >= date '${ods.cutoff_date}'
union all
select  4, 'HomeQS', 'Home', Ref, QuoteDateTime, CoverStartDate, QuotedPremium, AcceptedPremium, SourcePolicyCode, null
from    ${ods.stg}.odsv2_quote_home_cc where QuoteDateTime >= date '${ods.cutoff_date}'
union all
select  8, 'TravelQuotes', 'Travel', Ref, QuoteDateTime, CoverStartDate, QuotedPremium, null, null, CertificateReference
from    ${ods.stg}.odsv2_quote_travel where QuoteDateTime >= date '${ods.cutoff_date}'
union all
select  7, 'VanQuotes', 'Van', Ref, QuoteDateTime, CoverStartDate, null, null, null, null
from    ${ods.stg}.odsv2_quote_van where QuoteDateTime >= date '${ods.cutoff_date}'

In [ ]:
/******************************************************************************************************/
/*  Name        : {target}.Quote                                                                        */
/*  Description : Every quote across the quote systems: when it was created, cover start, the quoted an */
/*              : d accepted premium, and the                                                           */
/*              : policy it became. SourcePolicyCode is the code as the quote system holds it; PolicyId */
/*              :  is set only where that code                                                          */
/*              : is a real policy (in ods Policy) - the quote systems also carry non-policy references */
/*              : .                                                                                     */
/*              :                                                                                       */
/*              : Grain: one row per source system and source quote reference                           */
/*              : Business key: SourceSystemId + SourceSystemQuoteReference   Surrogate key: QuoteId    */
/*              : History: current   Cut-off: QuoteDateTime                                             */
/*              : Source: dlk.mfq_quotequery + edw.tbl_fact_mfq, dlk.ext_quotationstorage_*, dlk.hfq_*, */
/*              :  dlk.ext_home_qs_*, dlk.ext_travel_quotes, dlk.vanquotedetails                        */
/*              : Reads: stg.odsv2_quote_all, ods_v2.Policy                                             */
/******************************************************************************************************/
create or replace table ${ods.target}.Quote as
select  row_number() over (order by SourceSystemId, SourceSystemQuoteReference)as QuoteId,
        x.*,
        '${ods.batch_id}'                                           as OdsBatchId,
        current_timestamp()                                         as OdsLoadDateTime
from    (
        select  a.SourceSystemId,
                a.SourceSystem,
                a.ProductType,
                a.Ref                                                           as SourceSystemQuoteReference,
                a.QuoteDateTime,
                a.CoverStartDate,
                a.QuotedPremium,
                a.AcceptedPremium,
                coalesce(a.SourcePolicyCode, t.PolicyCode)                      as SourcePolicyCode,
                coalesce(p.PolicyId, t.PolicyId)                                as PolicyId,
                case when coalesce(p.PolicyId, t.PolicyId) is not null then 1 else 0 end
                                                                                as ConvertedToPolicy
        from    ${ods.stg}.odsv2_quote_all  a
        left join ${ods.target}.Policy      p   on p.PolicyCode = a.SourcePolicyCode
        left join (
                select  InsPolNumber, min(PolicyId) as PolicyId, min(PolicyCode) as PolicyCode
                from    ${ods.target}.Policy
                where   PolicyTypeGroup = 'Travel' and InsPolNumber is not null
                group by InsPolNumber
                ) t     on t.InsPolNumber = a.CertificateReference
        ) x

## PolicyQuoteLink


In [ ]:
/******************************************************************************************************/
/*  Name        : {target}.PolicyQuoteLink                                                              */
/*  Description : The warehouse's quote-to-policy matches (Relay quote events, with the MFQ guid or HEQ */
/*              :  quotation they matched),                                                             */
/*              : as notebook 02 stg.QuoteToPolicyLink. One row per quote event (one duplicate in the s */
/*              : ource: latest kept).                                                                  */
/*              :                                                                                       */
/*              : Grain: one row per warehouse quote event                                              */
/*              : Business key: QuoteEventID                                                            */
/*              : History: event   Cut-off: QuoteEventDateTime                                          */
/*              : Source: edw.tbl_fact_policy_quotes                                                    */
/*              : Reads: edw.tbl_fact_policy_quotes, ods_v2.Policy, ods_v2.Quote                        */
/******************************************************************************************************/
create or replace table ${ods.target}.PolicyQuoteLink as
select  x.*,
        '${ods.batch_id}'                                           as OdsBatchId,
        current_timestamp()                                         as OdsLoadDateTime
from    (
        select  x.QuoteEventID,
                p.PolicyId,
                qm.QuoteId                                                      as MfqQuoteId,
                x.QuoteEventDateTime,
                x.QuoteEventType,
                x.QuoteEventReportingTxnType,
                nullif(upper(trim(x.ClientCode)), '')                                              as ClientCode,
                nullif(upper(trim(x.PolicyCode)), '')                                              as PolicyCode,
                x.PolicyTypeCode,
                x.PolicyTypeGroup,
                x.PolicyStatusCode,
                x.RiskProfileBand,
                x.LeadType,
                x.OpportunityID,
                x.ChannelCall,
                x.ChannelWeb,
                x.ChannelXSell,
                x.ChannelOnLine,
                nullif(lower(trim(x.MFQQuoteQueryGUID)), '')                                       as MfqGuidKey,
                x.MFQCreatedDatetime,
                x.MFQQuoteType,
                x.MFQMatchType,
                x.MFQBrokerID,
                case when x.HEQQuotationID like '%@%' then ods_hash_id(x.HEQQuotationID, '')      -- some ids are email + timestamp
                     else nullif(upper(trim(x.HEQQuotationID)), '') end                            as HeqQuotationId,
                x.HEQCreatedDatetime,
                x.HEQQuoteType,
                x.HEQMatchType
        from    (
                select  *, row_number() over (partition by QuoteEventID order by CreateDate desc) as rn
                from    ${ods.edw}.tbl_fact_policy_quotes
                where   QuoteEventDateTime >= date '${ods.cutoff_date}'
                ) x
        left join ${ods.target}.Policy  p   on p.PolicyCode = upper(trim(x.PolicyCode))
        left join ${ods.target}.Quote   qm  on qm.SourceSystemId = 2 and qm.SourceSystemQuoteReference = nullif(lower(trim(x.MFQQuoteQueryGUID)), '')
        where   x.rn = 1
        ) x

## Checks


In [ ]:
%%pyspark
# Checks for the tables this notebook built. All are checked and logged to OdsBuildLog; the cell then fails if
# any check failed, naming the table and the likely cause.
import json, re
from functools import reduce
from pyspark.sql import functions as F
SPEC = json.loads(r"""[{"table": "Quote", "bk": ["SourceSystemId", "SourceSystemQuoteReference"], "sk": "QuoteId", "nullable": [], "allow": []}, {"table": "PolicyQuoteLink", "bk": ["QuoteEventID"], "sk": null, "nullable": [], "allow": []}]""")
PII_NAME = re.compile(r"(e_?mail|forename|first_?name|surname|last_?name|full_?name|second_?name|phone|mobile|telephone|post_?code|eircode(?!_?area)|address_?line|risk_?address|postal_?address|dob$|date_?of_?birth|licen[cs]e_?num|registration_?no|reg_?number|registration_number|^ani$|^dnis$|originatingdnis|^payer_|iban|^bic|^contact$|rendered_?message)", re.I)
EMAIL_RE = r"[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}"
MOBILE_RE = r"(^|[^0-9A-Za-z-])(\+?353 ?|0)8[35-9][ -]?[0-9]{3}[ -]?[0-9]{4}([^0-9A-Za-z-]|$)"   # stands alone, not inside an id
rows, failures = [], []
for s in SPEC:
    t = f"{target_schema}.{s['table']}"
    df = spark.table(t)
    bk = s["bk"]
    must = [c for c in bk if c not in s["nullable"]]
    agg = df.agg(F.count(F.lit(1)).alias("n"),
                 F.countDistinct(F.struct(*bk)).alias("d_bk"),
                 F.sum(F.when(reduce(lambda a, b: a | b, [F.col(c).isNull() for c in must] or [F.lit(False)]), 1).otherwise(0)).alias("null_bk"),
                 *( [F.countDistinct(s["sk"]).alias("d_sk")] if s["sk"] else [] )).first()
    pii_cols = [c for c in df.columns if PII_NAME.search(c) and not c.endswith("Key") and c not in s["allow"]]
    text_cols = [f.name for f in df.schema.fields if f.dataType.simpleString() == "string"]   # hashes never match
    hits = {}
    if text_cols:
        found = df.select(*text_cols).limit(1000000).agg(
            *[F.sum((F.col(c).rlike(EMAIL_RE) | F.col(c).rlike(MOBILE_RE)).cast("int")).alias(c) for c in text_cols]).first().asDict()
        hits = {c: v for c, v in found.items() if v}
    problems, advice = [], []
    keys = " + ".join(bk)
    if agg["n"] == 0:
        problems.append("empty")
        advice.append("the table is empty: its source is empty, or a filter (cut-off date, join) removed every row")
    if agg["d_bk"] != agg["n"]:
        problems.append(f"{agg['n'] - agg['d_bk']:,} duplicate business keys")
        advice.append(f"several rows share a key ({keys}): the source sent the same record more than once, or the "
                      f"dedupe rule in the cell no longer holds. Examples: select {', '.join(bk)}, count(*) from {t} "
                      f"group by {', '.join(bk)} having count(*) > 1")
    if agg["null_bk"]:
        problems.append(f"{agg['null_bk']:,} null business keys")
        advice.append(f"key columns ({', '.join(must)}) are empty on some rows: if that is legitimate for a column, add it "
                      f"to key_nullable for this table in the config; otherwise the source changed")
    if s["sk"] and agg["d_sk"] != agg["n"]:
        problems.append("surrogate key not unique")
        advice.append("the surrogate key repeats: the business key is not unique (see above)")
    if pii_cols:
        problems.append(f"PII-named columns {pii_cols}")
        advice.append("columns named like personal data: hash them (HASH_*) or leave them out in the config")
    if hits:
        problems.append(f"email/mobile values in {hits}")
        advice.append("emails / mobile numbers inside these columns: wrap the column in REDACT() (or hash it) in the config")
    rows.append((batch_id, s["table"], agg["n"], agg["n"] - agg["d_bk"], agg["null_bk"], "; ".join(problems) or "ok"))
    print(f"{s['table']:<28} {agg['n']:>13,}  {'; '.join(problems) or 'ok'}")
    if problems:
        failures.append(f"{s['table']}: " + " | ".join(advice))
log = spark.createDataFrame(rows, "OdsBatchId string, TableName string, RowCount long, DuplicateKeys long, NullKeys long, Result string")
log.withColumn("CheckedAt", F.current_timestamp()).write.mode("append").saveAsTable(f"{target_schema}.OdsBuildLog")
if failures:
    ods_run_log("20_Quote", "failed", "; ".join(f.split(":")[0] for f in failures))
    raise AssertionError("ods checks failed:\n- " + "\n- ".join(failures))
ods_run_log("20_Quote", "succeeded")
